# Customer Churn Prediction – Exploratory Data Analysis (EDA)
**Project:** ChurnGuard – Customer Retention Intelligence  
**Dataset:** IBM Telco Customer Churn (7,043 customer records)  
**Objective:** Analyze customer behavior, identify churn drivers, and prepare features for ML modeling.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
%matplotlib inline

# Load raw data
df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")
print("Dataset Shape:", df.shape)
df.head()

## 1. Data Cleaning & Type Formatting
- Convert `TotalCharges` to numeric.
- Impute missing `TotalCharges` for zero-tenure customers with `0.0`.
- Standardize redundant service categories (`No internet service` -> `No`).
- Drop non-predictive `customerID`.

In [2]:
# Convert TotalCharges
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df.loc[df["tenure"] == 0, "TotalCharges"] = 0.0
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

# Standardize redundant categories
redundant_cols = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]
for col in redundant_cols:
    df[col] = df[col].replace({"No internet service": "No", "No phone service": "No"})

df = df.drop(columns=["customerID"]).drop_duplicates()
print("Cleaned shape:", df.shape)

## 2. Overall Churn Distribution

In [3]:
churn_counts = df["Churn"].value_counts(normalize=True) * 100
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df, x="Churn", palette=["#2b5c8f", "#e74c3c"])
plt.title("Overall Churn Distribution")
plt.ylabel("Number of Customers")
plt.show()
print("Churn Percentage:", churn_counts.to_dict())

## 3. Churn Analysis by Contract Type
Month-to-month contracts exhibit drastically higher churn compared to one-year and two-year commitments.

In [4]:
plt.figure(figsize=(8, 4))
contract_churn = df.groupby("Contract")["Churn"].apply(lambda x: (x == "Yes").mean() * 100).reset_index()
sns.barplot(data=contract_churn, x="Contract", y="Churn", palette="Blues_r")
plt.title("Churn Rate by Contract Commitment Type (%)")
plt.ylabel("Churn Rate (%)")
plt.show()

## 4. Churn Analysis by Internet Service & Payment Method

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
internet_churn = df.groupby("InternetService")["Churn"].apply(lambda x: (x == "Yes").mean() * 100).reset_index()
sns.barplot(ax=axes[0], data=internet_churn, x="InternetService", y="Churn", palette="viridis")
axes[0].set_title("Churn Rate by Internet Service Type (%)")
axes[0].set_ylabel("Churn Rate (%)")

payment_churn = df.groupby("PaymentMethod")["Churn"].apply(lambda x: (x == "Yes").mean() * 100).reset_index()
sns.barplot(ax=axes[1], data=payment_churn, x="PaymentMethod", y="Churn", palette="magma")
axes[1].set_title("Churn Rate by Payment Method (%)")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha="right")
axes[1].set_ylabel("Churn Rate (%)")
plt.tight_layout()
plt.show()

## 5. Tenure & Monthly Charges Distributions

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.kdeplot(ax=axes[0], data=df, x="tenure", hue="Churn", common_norm=False, fill=True, palette=["#2b5c8f", "#e74c3c"])
axes[0].set_title("Customer Tenure Density: Retained vs Churned")
axes[0].set_xlabel("Tenure (Months)")

sns.boxplot(ax=axes[1], data=df, x="Churn", y="MonthlyCharges", palette=["#2b5c8f", "#e74c3c"])
axes[1].set_title("Monthly Charges Distribution by Churn")
plt.tight_layout()
plt.show()

## 6. Correlation Heatmap & Customer Segments

In [7]:
df_encoded = df.copy()
df_encoded["Churn_Binary"] = (df_encoded["Churn"] == "Yes").astype(int)
numeric_df = df_encoded.select_dtypes(include=[np.number])

plt.figure(figsize=(8, 6))
sns.heatmap(numeric_df.corr(), annot=True, cmap="coolwarm", fmt=".2f", cbar=True)
plt.title("Correlation Matrix of Numeric Features and Churn")
plt.tight_layout()
plt.show()

## 7. Strategic Conclusions
1. **Month-to-month contracts** are the single highest churn risk factor (~42% churn vs <3% for 2-year contracts).
2. **Fiber optic customers** experience significantly higher churn than DSL (~41% vs 19%), driven by high monthly rates and lack of tech support.
3. **Electronic check** users churn at ~45%, compared to ~16% for automatic bank transfer or credit card users.
4. **Early lifecycle (< 12 months)** is the most fragile tenure window, necessitating concierge onboarding.